# DiCo-NLI - Track 1, frozen-backbone sweep (ModernBERT / NLI-pretrained mDeBERTa)

**Repository:** https://github.com/AmnO-O/DICO-NLI

Fine-tunes one backbone on **track 1 (English) only**, freezing part of the encoder. Track 1's
train split is only **3,042 rows**, so full fine-tuning a large encoder mostly memorises.

Two backbones are wired up:

- `answerdotai/ModernBERT-base` - raw masked-LM encoder, 22 layers, 149M params.
- `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` - **already NLI-fine-tuned** on MNLI/XNLI.
  12 layers, 251k multilingual vocab, 279M params of which **193M (69%) are the embedding
  table**, so the grid keeps embeddings frozen and moves only the encoder: 30.9% / 15.6% / 8.0%
  trainable at freeze 0 / 6 / 9. On 3k labelled rows that pretraining is worth more than size.

Both checkpoints need care on `transformers>=5`: the mDeBERTa file is stored in **fp16** and has
a **3-way** NLI head. The loader now forces fp32 (this pipeline has no autocast/GradScaler) and
reinitialises the head to 4 labels instead of raising on the size mismatch.

`--symmetric` scores every pair as both `(text1, text2)` and `(text2, text1)`, flips the reversed
labels and averages. Equal weights make the result **exactly reversal-equivariant**, so SoftCons
becomes 1.000 by construction instead of something the model has to learn.

## What is frozen (ModernBERT numbers)

| Freed | Frozen |
|-------|--------|
| top `22 - N` encoder layers | bottom `N` encoder layers |
| pooler + `final_norm` | **input embeddings** (`tok_embeddings`, 38.7M params) |
| classifier head (768 -> 4) | |

`FREEZE_EMBEDDINGS = True` matters: on ModernBERT-base the embedding table is
`50368 x 768 = 38.7M` parameters, i.e. **~26% of the model**. If you leave it trainable, even
`--freeze_bottom_layers 22` still trains ~27% of all weights.

Resulting trainable share on ModernBERT-base (22 layers):

| `FREEZE_LAYERS` | Trainable | What that means |
|-----------------|-----------|-----------------|
| `12` | 50,745,604 (33.9%) | top 10 layers |
| `14` | 40,715,524 (27.2%) | top 8 layers |
| `16` | 30,685,444 (20.5%) | top 6 layers |
| `18` | 20,655,364 (13.8%) | top 4 layers |
| `20` | 10,625,284 (7.1%) | top 2 layers |
| `21` | 5,610,244 (3.8%) | top 1 layer |
| `22` | 595,204 (0.4%) | pure linear probe |

Measured on track 1 so far (ModernBERT, 660-row dev, +/-1.8 F1 noise):

| freeze | trainable | best dev F1 | best epoch | soft | hard |
|--------|-----------|-------------|------------|------|------|
| `16` (top 6) | 20.5% | 0.6840 | 4 | 0.773 | 0.686 |
| `17` (top 5) | 17.2% | 0.6985 | 10 | 0.729 | 0.650 |

Freezing `19+` (top 3) underfits: train loss stalls at 1.33 = chance. Freezing `16-17` saturates
around **0.68-0.70** and train loss collapses to ~0.001 by epoch 6 - the model memorises the
3,042 rows long before dev improves. Every run exports its **best dev epoch**, not its last one.

## Where the next points come from

`0.68-0.70` is still ~12 F1 behind a *single* NLI-pretrained mDeBERTa (0.818 on the same dev).
That gap is not epochs or dropout; it is (a) NLI pretraining and (b) directional symmetry.
This notebook now tests both: `ACTIVE_MODELS` switches the backbone to
`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`, and `SYMMETRIC = True` makes predictions exactly
reversal-consistent. The structural prior (a pair with a reversed twin is reversible) is the
third lever and is applied post hoc, outside this notebook.

## Speed: `FAST = True`

The starter kit's `set_seed` sets `CUDA_LAUNCH_BLOCKING=1`, `torch.use_deterministic_algorithms(True)`
and `FLASH_ATTENTION_DETERMINISTIC=1`. Each of those forces a **device sync per kernel launch**. At
`max_length = 256` a kernel runs long enough that the sync is hidden; at **64** it is not, and the
step time becomes launch-bound instead of compute-bound (measured: 46 s/epoch at 64 tokens vs
41 s/epoch at 256, i.e. 4x less arithmetic for the same wall clock).

`--fast` skips those guards. Seeding is unchanged, so seeds still control everything that matters;
the only thing lost is bit-exact reproducibility across some CUDA kernels.

## Sequence length

`MAX_LENGTH = 64` is enough for **all four tracks**: with both the ModernBERT and the mmBERT
tokenizer, the longest training row is 59 tokens (track 3, Basque) and the longest dev row is 48,
so **0.00%** of rows are truncated. Dropping from 256 to 64 is a ~4x speed-up on identical data
(at 256 more than 96% of every batch is padding: the track-1 median is 10 tokens).

## How it works

Each run is a subprocess of `starter_kit/main.py` (the documented pipeline), so this reproduces the
starter-kit protocol exactly. After each run the notebook:

1. picks the dev prediction file of the **best dev-F1 epoch** (`predictions_dev_<timestamp>.csv`;
   the last-epoch score is recorded next to it, so overfitting is visible as a gap),
2. scores it against `<DATA_ROOT>/dev/dico_nli_dev_track1_reference.csv` with the official scorer,
3. appends `weighted_f1`, `soft_cons`, `hard_cons` (+ how many parameters were actually trainable)
   to a running CSV.

Runs are **resumable**: re-running skips any setting that already has predictions on disk.
Set `OVERWRITE = True` to force a clean re-run.

## Data location (auto-detected)

- **`REPO_ROOT`** - the code checkout (`starter_kit/`, `evaluation_functions/`).
- **`DATA_ROOT`** - the data, which must contain `train/` and `dev/`.

| Environment | `DATA_ROOT` |
|-------------|-------------|
| Kaggle | `/kaggle/input/datasets/ieltsmater/dico-nli/DiCO-NLI` |
| Local checkout | `<repo>/final_data` |

## Running on Kaggle

1. Add the `ieltsmater/dico-nli` dataset to the notebook.
2. Enable **Internet** (Settings -> Internet on) so the setup cell can `git clone` the repo.
3. GPU **T4 x2 / P100** is enough; run all cells and results land in
   `/kaggle/working/results/modernbert_track1_frozen`.

> The checkout must be at commit `f08b1e5` or later: that is where `--freeze_embeddings`,
> `--patience`, `--fast` and `--symmetric` exist. The setup cell prints the commit it is running.


## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
import re
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:  # plain-python fallback (nbconvert / script)
    def display(obj):
        print(obj)


GITHUB_REPO = "https://github.com/AmnO-O/DICO-NLI"
GITHUB_BRANCH = "main"

# Kaggle paths (only active when running on Kaggle's Linux environment).
KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_DATA_ROOT = Path("/kaggle/input/datasets/ieltsmater/dico-nli/DiCO-NLI")
IS_KAGGLE = sys.platform.startswith("linux") and KAGGLE_WORKING.is_dir()

# Set to a path to force a specific data directory, or leave None to auto-detect.
DATA_ROOT_OVERRIDE = None


def is_repo_root(path: Path) -> bool:
    return (path / "starter_kit" / "main.py").exists() and (path / "evaluation_functions").is_dir()


def find_repo_root(start: Path):
    for candidate in [start, *start.parents]:
        if is_repo_root(candidate):
            return candidate
    return None


def ensure_repo() -> Path:
    # 1) a checkout is already available (local machine, or bundled in the Kaggle dataset)
    candidates = [Path.cwd().resolve()]
    if IS_KAGGLE:
        candidates += [KAGGLE_DATA_ROOT, KAGGLE_WORKING / "DiCO-NLI"]
    for candidate in candidates:
        if candidate.exists():
            root = find_repo_root(candidate)
            if root is not None:
                return root
    # 2) otherwise clone it (Kaggle: enable Internet in the notebook settings)
    clone_dir = (KAGGLE_WORKING if IS_KAGGLE else Path.cwd()) / "DiCO-NLI"
    if not is_repo_root(clone_dir):
        print(f"Cloning {GITHUB_REPO} -> {clone_dir}")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(clone_dir)],
            check=True,
        )
    return clone_dir


def resolve_data_root() -> Path:
    if DATA_ROOT_OVERRIDE is not None:
        return Path(DATA_ROOT_OVERRIDE)
    if IS_KAGGLE and (KAGGLE_DATA_ROOT / "train").is_dir() and (KAGGLE_DATA_ROOT / "dev").is_dir():
        return KAGGLE_DATA_ROOT
    return REPO_ROOT / "final_data"


REPO_ROOT = ensure_repo()
STARTER_KIT = REPO_ROOT / "starter_kit"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_functions import score_files  # noqa: E402

DATA_ROOT = resolve_data_root()

def repo_version(root: Path) -> str:
    # Which commit is actually being executed? On Kaggle ensure_repo() can pick
    # the bundled dataset copy instead of a fresh clone, so this matters.
    try:
        out = subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                             capture_output=True, text=True, encoding="utf-8", errors="replace")
        return out.stdout.strip() or "unknown (not a git checkout)"
    except Exception:
        return "unknown"


print("Python  :", sys.executable)
print("Kaggle  :", IS_KAGGLE)
print("Repo    :", REPO_ROOT, "| git", repo_version(REPO_ROOT))
print("GitHub  :", GITHUB_REPO)
print("Data    :", DATA_ROOT)
print("Train   :", (DATA_ROOT / "train").exists())
print("Dev     :", (DATA_ROOT / "dev").exists())


def run_logged_command(cmd, cwd, log_path,
                       echo=("Epoch", "average loss", "Training history", "Trainable parameters",
                             "F1-weighted", "Early stopping")):
    # Run a command, tee its stdout/stderr into log_path, and echo the lines that
    # matter (freeze confirmation, per-epoch loss, per-epoch dev F1) live so you can
    # watch progress. The dev set IS evaluated after every epoch; without
    # "F1-weighted" in this tuple those lines would only land in the log file.
    with Path(log_path).open("w", encoding="utf-8") as log:
        log.write(" ".join(cmd) + "\n\n")
        log.flush()
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        for line in proc.stdout:
            log.write(line)
            log.flush()
            if any(token in line for token in echo):
                print("    " + line.rstrip())
        returncode = proc.wait()
    return returncode


In [ ]:
# ----------------------------- experiment grid -----------------------------
MODELS = {
    "ModernBERT-base": "answerdotai/ModernBERT-base",
    "ModernBERT-large": "answerdotai/ModernBERT-large",
    "mDeBERTa-v3-base-mnli-xnli": "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
}
# Encoder depth per backbone: used for the freeze range and the printout.
MODEL_LAYERS = {
    "ModernBERT-base": 22,
    "ModernBERT-large": 28,
    "mDeBERTa-v3-base-mnli-xnli": 12,
}
ACTIVE_MODELS = ["mDeBERTa-v3-base-mnli-xnli"]
TRACK = 1                             # English monolingual only
SEEDS = [42]                          # one seed first; expand to 3 once the backbone question is settled

# mDeBERTa is NLI-pretrained, so the question is not "can it learn the task" but
# "how much of the pretrained stack should move on 3k rows".
# Measured on the real checkpoint (278,812,420 params; embeddings are 193M = 69%):
#   freeze 0 -> all 12 layers + head   86.0M (30.9%)
#   freeze 6 -> top 6 layers + head    43.5M (15.6%)
#   freeze 9 -> top 3 layers + head    22.3M ( 8.0%)
FREEZE_VALUES = [0, 6, 9]

# Without this, freeze=6 still trains 236M (84.7%) because the 193M multilingual
# embedding table stays trainable, so the freeze knob barely does anything.
FREEZE_EMBEDDINGS = True

# ----------------------------- hyperparameters -----------------------------
# These must satisfy starter_kit/main.py validation rules.
MAX_LENGTH = 64    # longest row on any track is 59 tokens -> 0 rows truncated
NUM_EPOCHS = 10    # 1..100 ; with PATIENCE set this is only an upper bound, not a target
LR = 3e-5          # NLI-pretrained backbone: 1e-4 is too high, 3e-5 is the standard full-FT range
BATCH_SIZE = 32    # multiple of 8, 8 <= batch_size <= 64
WD = 1e-5          # allowed: 1e-6, 5e-6, 8e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1
WARMUP_PCT = 0.01  # 0 .. 0.1, step 0.01
PATIENCE = 3       # 660-row dev has ~+/-1.8 F1 noise; patience 2 stops on a dip
FAST = True        # skip CUDA_LAUNCH_BLOCKING + deterministic kernels (big win at max_length 64)
DROPOUT = 0.1      # 0 .. 0.5 (classifier-head dropout; DeBERTa's encoder dropout stays at config)

# ------------------------- reversal-equivariant inference -------------------
# Score (a, b) and (b, a), flip FORWARD<->BACKWARD on the reversed view, average
# with EQUAL weights. That is what makes the result exactly reversal-consistent;
# any weighting other than 0.5 breaks the guarantee.
SYMMETRIC = True
SYMMETRIC_MODE = "prob"   # prob | logprob | logit - try each on dev and keep the best

# ----------------------------- output locations ----------------------------
# On Kaggle write under /kaggle/working so results are kept as notebook output.
OUTPUT_ROOT = KAGGLE_WORKING if IS_KAGGLE else REPO_ROOT
RESULTS_DIR = OUTPUT_ROOT / "results" / "modernbert_track1_frozen"
LOG_DIR = RESULTS_DIR / "logs"
SUMMARY_CSV = RESULTS_DIR / "frozen_scores.csv"
OVERWRITE = False      # True -> delete and re-run completed experiments
SAVE_MODELS = False    # True -> also save each fine-tuned checkpoint (~600 MB per run)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# starter_kit/main.py rejects these values with an AssertionError. Check them here
# so a typo fails in one second instead of after the model download.
ALLOWED_LR = [1e-6, 5e-6, 8e-6, 9e-6, 1e-5, 1.5e-5, 2e-5, 2.5e-5, 3e-5, 4e-5, 5e-5, 8e-5, 1e-4]
ALLOWED_WD = [1e-6, 5e-6, 8e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1]
assert 1 <= NUM_EPOCHS <= 100, f"NUM_EPOCHS={NUM_EPOCHS} must be between 1 and 100"
assert 8 <= BATCH_SIZE <= 64 and BATCH_SIZE % 8 == 0, f"BATCH_SIZE={BATCH_SIZE} must be 8..64, multiple of 8"
assert LR in ALLOWED_LR, f"LR={LR} must be one of {ALLOWED_LR}"
assert WD in ALLOWED_WD, f"WD={WD} must be one of {ALLOWED_WD}"
assert 0 <= DROPOUT <= 0.5, f"DROPOUT={DROPOUT} must be 0..0.5"
assert 0 <= WARMUP_PCT <= 0.1, f"WARMUP_PCT={WARMUP_PCT} must be 0..0.1"
assert PATIENCE is None or PATIENCE >= 1, f"PATIENCE={PATIENCE} must be >= 1 or None"
assert SYMMETRIC_MODE in ("prob", "logprob", "logit"), f"SYMMETRIC_MODE={SYMMETRIC_MODE}"
assert all(0 <= freeze <= MODEL_LAYERS[m] for m in ACTIVE_MODELS for freeze in FREEZE_VALUES), \
    "FREEZE_VALUES must be within each active model's layer count"

TOTAL = len(ACTIVE_MODELS) * len(FREEZE_VALUES) * len(SEEDS)
print(f"Configured {TOTAL} experiments (track {TRACK}) -> {RESULTS_DIR}")
for model_key in ACTIVE_MODELS:
    n_layers = MODEL_LAYERS[model_key]
    for freeze in FREEZE_VALUES:
        print(f"  {model_key}: freeze={freeze:>2}/{n_layers} -> train top {n_layers - freeze} layer(s) + head"
              f"{' (embeddings frozen)' if FREEZE_EMBEDDINGS else ''}")
    print(f"  {model_key}: symmetric={SYMMETRIC} ({SYMMETRIC_MODE})")


## 2. Helper functions

In [ ]:
def track_paths(track: int):
    train = DATA_ROOT / "train" / f"dico_nli_train_track{track}_participant_labeled.csv"
    dev = DATA_ROOT / "dev" / f"dico_nli_dev_track{track}_participant_labeled.csv"
    ref = DATA_ROOT / "dev" / f"dico_nli_dev_track{track}_reference.csv"
    for path in (train, dev, ref):
        if not path.exists():
            raise FileNotFoundError(path)
    return train, dev, ref


def run_key(model_key: str, freeze: int, seed: int) -> str:
    slug = model_key.lower().replace(".", "")
    # Symmetric runs get their own directory: without the suffix they would be
    # skipped as "already scored" by the non-symmetric run of the same config.
    suffix = "_sym" if SYMMETRIC else ""
    return f"{slug}_track{TRACK}_f{freeze}_seed{seed}{suffix}"


def build_command(model_id, seed, results_dir, freeze, *, save_path=None):
    train, dev, ref = track_paths(TRACK)
    return [
        sys.executable, "-u", "main.py",
        "--model_path", model_id,
        "--tokenizer_path", model_id,
        "--do_train",
        "--train_path", str(train),
        "--do_dev",
        "--dev_path", str(dev),
        "--no-do_test",
        "--seed", str(seed),
        "--dropout", str(DROPOUT),
        "--warmup_pcrt", str(WARMUP_PCT),
        "--lr", str(LR),
        "--batch_size", str(BATCH_SIZE),
        "--wd", str(WD),
        "--num_epochs", str(NUM_EPOCHS),
        *(["--patience", str(PATIENCE)] if PATIENCE is not None else []),
        *(["--fast"] if FAST else []),
        *(["--symmetric", "--symmetric_mode", SYMMETRIC_MODE] if SYMMETRIC else []),
        "--no-is_optuna_trial",
        "--max_length", str(MAX_LENGTH),
        "--results_dir", str(results_dir),
        "--reference_path", str(ref),
        "--freeze_bottom_layers", str(freeze),
        *(["--freeze_embeddings"] if FREEZE_EMBEDDINGS else []),
        *(["--save_path", str(save_path)] if save_path is not None else []),
    ]


TRAINABLE_RE = re.compile(r"Trainable parameters:\s*([\d,]+)\s*/\s*([\d,]+)\s*\(([\d.]+)%\)")


def parse_trainable(log_path: Path):
    # Read the 'Trainable parameters: A / B (P%)' line printed by model.py.
    if not log_path.exists():
        return None, None
    text = log_path.read_text(encoding="utf-8", errors="replace")
    match = TRAINABLE_RE.search(text)
    if match is None:
        return None, None
    trainable = int(match.group(1).replace(",", ""))
    total = int(match.group(2).replace(",", ""))
    return trainable, round(100.0 * trainable / total, 2)


def score_predictions(key, model_key, model_id, freeze, seed, pred_path, ref_path, run_dir, elapsed=None):
    report = score_files(ref_path, pred_path, output_dir=run_dir / "official_scores" / pred_path.stem)
    trainable, trainable_pct = parse_trainable(LOG_DIR / f"{key}.log")
    return {
        "run": key,
        "model": model_key,
        "model_id": model_id,
        "track": TRACK,
        "freeze_layers": freeze,
        "freeze_embeddings": FREEZE_EMBEDDINGS,
        "symmetric": SYMMETRIC,
        "symmetric_mode": SYMMETRIC_MODE,
        "seed": seed,
        "weighted_f1": report.weighted_f1,
        "soft_cons": report.soft_cons,
        "hard_cons": report.hard_cons,
        "macro_f1": report.classification.macro_f1,
        "accuracy": report.classification.accuracy,
        "instances": report.classification.total_instances,
        "trainable_params": trainable,
        "trainable_pct": trainable_pct,
        "predictions": str(pred_path),
        "minutes": None if elapsed is None else round(elapsed / 60, 2),
    }


def read_history(run_dir: Path):
    """Per-epoch records the trainer rewrites after every epoch (dev F1 included)."""
    path = run_dir / "training_history.json"
    if not path.exists():
        return []
    try:
        history = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return []
    return [row for row in history if "dev_f1_weighted" in row]


def best_epoch_of(history):
    """1-based epoch with the highest dev F1, or None when unavailable."""
    if not history:
        return None
    return int(max(history, key=lambda row: row["dev_f1_weighted"])["epoch"])


def score_run(key, model_key, model_id, freeze, seed, run_dir, elapsed=None):
    """Score the run at its **last** epoch and at its **best dev** epoch.

    The trainer writes one ``predictions_dev_*.csv`` per epoch, so the last one is
    not necessarily the best. ``weighted_f1`` / ``soft_cons`` / ``hard_cons`` stay
    last-epoch numbers (comparable with runs scored before this was tracked);
    ``weighted_f1_best_epoch`` is what section 10 exports.
    """
    epoch_preds = sorted(run_dir.glob("predictions_dev_*.csv"))
    if not epoch_preds:
        raise FileNotFoundError(f"{key}: no predictions_dev_*.csv in {run_dir}")

    ref_path = track_paths(TRACK)[2]
    row = score_predictions(key, model_key, model_id, freeze, seed,
                            epoch_preds[-1], ref_path, run_dir, elapsed)

    history = read_history(run_dir)
    if not history:
        return row

    best_epoch = best_epoch_of(history)
    row["best_epoch"] = best_epoch
    # True when the best epoch is the last one: the run was still improving and
    # the epoch budget, not overfitting, decided where training stopped.
    row["best_is_last"] = best_epoch == len(epoch_preds)
    row["dev_f1_best"] = float(max(h["dev_f1_weighted"] for h in history))
    row["dev_f1_last"] = float(history[-1]["dev_f1_weighted"])
    row["train_loss_last"] = float(history[-1]["train_loss"])

    if not 1 <= best_epoch <= len(epoch_preds):
        print(f"warning: {key}: best epoch {best_epoch} has no prediction file "
              f"({len(epoch_preds)} found) - keeping the last-epoch score")
        return row

    best_pred = epoch_preds[best_epoch - 1]
    row["best_epoch_predictions"] = str(best_pred)
    if best_pred == epoch_preds[-1]:
        row["weighted_f1_best_epoch"] = row["weighted_f1"]
        row["soft_cons_best_epoch"] = row["soft_cons"]
        row["hard_cons_best_epoch"] = row["hard_cons"]
    else:
        scored = score_predictions(key, model_key, model_id, freeze, seed,
                                   best_pred, ref_path, run_dir)
        row["weighted_f1_best_epoch"] = scored["weighted_f1"]
        row["soft_cons_best_epoch"] = scored["soft_cons"]
        row["hard_cons_best_epoch"] = scored["hard_cons"]
    return row


def run_experiment(model_key, model_id, freeze, seed, *, overwrite=OVERWRITE):
    key = run_key(model_key, freeze, seed)
    run_dir = RESULTS_DIR / key
    log_path = LOG_DIR / f"{key}.log"

    if run_dir.exists() and not overwrite and sorted(run_dir.glob("predictions_dev_*.csv")):
        print(f"[skip] {key} -> scoring existing predictions")
        return score_run(key, model_key, model_id, freeze, seed, run_dir)

    if overwrite and run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    save_path = run_dir / "model" if SAVE_MODELS else None
    cmd = build_command(model_id, seed, run_dir, freeze, save_path=save_path)
    print(f"[run ] {key}")
    started = time.time()
    returncode = run_logged_command(cmd, STARTER_KIT, log_path)
    elapsed = time.time() - started

    if returncode != 0:
        print(f"[fail] {key}: exit {returncode}")
        print(f"---- last 30 lines of {log_path.name} ----")
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-30:]
        for line in tail:
            print("    " + line)
        print("---- end of log ----")
        return None

    n_epochs = len(list(run_dir.glob("predictions_dev_*.csv")))
    print(f"[ok  ] {key} in {elapsed / 60:.1f} min -> {n_epochs} epoch prediction files")
    return score_run(key, model_key, model_id, freeze, seed, run_dir, elapsed)


## 3. Run the experiment grid

In [ ]:
if SUMMARY_CSV.exists() and not OVERWRITE:
    results = pd.read_csv(SUMMARY_CSV).to_dict("records")
    done = {(r["model"], int(r["freeze_layers"]), int(r["seed"]), bool(r.get("symmetric", False)))
            for r in results}
else:
    results, done = [], set()

print(f"{len(done)}/{TOTAL} experiments already scored\n")

for model_key in ACTIVE_MODELS:
    model_id = MODELS[model_key]
    for freeze in FREEZE_VALUES:
        for seed in SEEDS:
            if (model_key, freeze, seed, SYMMETRIC) in done:
                continue
            try:
                row = run_experiment(model_key, model_id, freeze, seed)
            except Exception:
                traceback.print_exc()
                row = None
            if row is not None:
                results.append(row)
                done.add((model_key, freeze, seed, SYMMETRIC))
                pd.DataFrame(results).to_csv(SUMMARY_CSV, index=False)
                gain = row.get("weighted_f1_best_epoch")
                gain = 0.0 if gain is None else gain - row["weighted_f1"]
                note = "  <- best epoch is the last one: raise NUM_EPOCHS" if row.get("best_is_last") else ""
                print(
                    f"     -> weighted_f1={row['weighted_f1']:.4f}  "
                    f"soft_cons={row['soft_cons']:.4f}  hard_cons={row['hard_cons']:.4f}  "
                    f"trainable={row['trainable_pct']}%  "
                    f"best_epoch={row.get('best_epoch')} (+{gain:.4f} vs last){note}"
                )

df = pd.DataFrame(results)
print(f"\nFinished: {len(df)}/{TOTAL} experiments scored.")


## 4. Aggregated results

In [ ]:
if not SUMMARY_CSV.exists():
    raise FileNotFoundError(f"No results yet at {SUMMARY_CSV} - run section 3 first.")

df = pd.read_csv(SUMMARY_CSV)
for col in ("track", "seed", "freeze_layers"):
    df[col] = df[col].astype(int)

# Columns added when best-epoch tracking landed; older rows do not have them.
for col in ("dev_f1_best", "dev_f1_last", "train_loss_last", "best_epoch",
            "weighted_f1_best_epoch", "best_is_last"):
    if col not in df:
        df[col] = float("nan")

summary = (
    df.groupby(["model", "freeze_layers"], as_index=False)
    .agg(
        n=("seed", "count"),
        trainable_pct=("trainable_pct", "max"),
        weighted_f1_last_mean=("weighted_f1", "mean"),
        weighted_f1_best_mean=("weighted_f1_best_epoch", "mean"),
        best_epoch_mean=("best_epoch", "mean"),
        budget_limited_share=("best_is_last", "mean"),
        train_loss_last_mean=("train_loss_last", "mean"),
        soft_cons_mean=("soft_cons", "mean"),
        soft_cons_std=("soft_cons", "std"),
        hard_cons_mean=("hard_cons", "mean"),
        hard_cons_std=("hard_cons", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        accuracy_mean=("accuracy", "mean"),
        minutes_mean=("minutes", "mean"),
    )
    .sort_values(["model", "freeze_layers"])
)
summary.to_csv(RESULTS_DIR / "frozen_summary.csv", index=False)
print(f"Track {TRACK} - mean over seeds (last epoch vs best dev epoch):")
display(summary.round(4))
print("\nA large (best - last) gap means the run overfits before the epoch budget ends: "
      "lower NUM_EPOCHS or raise DROPOUT.")


In [ ]:
# Freeze setting vs official metrics (mean over seeds).
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for model_key, part in summary.groupby("model"):
        for ax, metric in zip(axes, ("weighted_f1", "soft_cons", "hard_cons")):
            ax.errorbar(
                part["freeze_layers"],
                part[f"{metric}_mean"],
                yerr=part[f"{metric}_std"].fillna(0),
                marker="o", capsize=4, label=model_key,
            )
    for ax, metric, title in zip(
        axes,
        ("weighted_f1", "soft_cons", "hard_cons"),
        ("Weighted F1", "SoftCons", "HardCons"),
    ):
        ax.set_xlabel("freeze_bottom_layers (higher = more frozen)")
        ax.set_ylabel(title)
        ax.set_title(f"Track {TRACK}: {title}")
        ax.set_xticks(sorted(summary["freeze_layers"].unique()))
        ax.grid(alpha=0.3)
    axes[0].legend()
    fig.tight_layout()
    plt.show()
except ImportError:
    print("(matplotlib not installed - use the table above)")


In [ ]:
# Per-seed view, so you can judge whether a difference is real or just dev noise.
# The dev split is only 660 rows, so expect roughly +/-2 F1 points between seeds.
for metric in ("weighted_f1", "hard_cons"):
    print(f"{metric} by seed:")
    display(
        df.pivot_table(index=["model", "freeze_layers"], columns="seed", values=metric).round(4)
    )


## 5. Verify the freezing actually applied

Each run logs `Trainable parameters: A / B (P%)` and `Input embeddings frozen.`. If `trainable_pct` here does not match `1 - 22/22`, the freeze did not take effect.

In [ ]:
verify = (
    df.groupby(["model", "freeze_layers"], as_index=False)
    .agg(trainable_params=("trainable_params", "max"),
         trainable_pct=("trainable_pct", "max"),
         freeze_embeddings=("freeze_embeddings", "max"))
    .sort_values(["model", "freeze_layers"])
)
display(verify)

if verify["trainable_params"].isna().any():
    print("WARNING: some runs have no 'Trainable parameters' line in their log.")
if verify["trainable_pct"].dropna().max() > 30:
    print("WARNING: >30% of weights are trainable - embeddings are probably NOT frozen.")

sample_log = LOG_DIR / f"{df.iloc[0]['run']}.log"
lines = sample_log.read_text(encoding="utf-8", errors="replace").splitlines()
print(f"\nFreeze-related lines from {sample_log.name}:")
print("\n".join(l for l in lines if "rozen" in l or "Trainable parameters" in l))


## 6. Training curves (per-epoch loss and dev F1)

In [ ]:
# Every run writes <run_dir>/training_history.json (and .csv).
def load_history(run_dir):
    path = Path(run_dir) / "training_history.json"
    if not path.exists():
        return None
    return pd.DataFrame(json.loads(path.read_text(encoding="utf-8")))


available = sorted(RESULTS_DIR.glob("*/training_history.json"))
print(f"{len(available)} run(s) with a recorded training history")
for path in available:
    print(" -", path.parent.name)


In [ ]:
# One line per (freeze setting, seed): watch the dev F1 plateau/overfit point.
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    colors = {f: c for f, c in zip(sorted(df["freeze_layers"].unique()),
                                   ("tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"))}
    for path in available:
        hist = load_history(path.parent)
        if hist is None or hist.empty:
            continue
        freeze = int(re.search(r"_f(\d+)_", path.parent.name).group(1))
        axes[0].plot(hist["epoch"], hist["train_loss"], color=colors[freeze], alpha=0.6, marker="o", ms=3)
        if "dev_f1_weighted" in hist.columns:
            axes[1].plot(hist["epoch"], hist["dev_f1_weighted"], color=colors[freeze], alpha=0.6, marker="o", ms=3)
    for ax, title, ylabel in zip(axes, ("Train loss", "Dev weighted F1"), ("train loss", "weighted F1")):
        ax.set_title(f"{title} (colour = freeze_bottom_layers)")
        ax.set_xlabel("epoch")
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)
    handles = [plt.Line2D([], [], color=c, label=f"freeze={f}") for f, c in colors.items()]
    axes[1].legend(handles=handles, fontsize=8)
    fig.tight_layout()
    plt.show()
except ImportError:
    print("(matplotlib not installed)")

# Did the best epoch come last, or did dev F1 peak earlier and then degrade?
rows = []
for path in available:
    hist = load_history(path.parent)
    if hist is None or "dev_f1_weighted" not in hist.columns:
        continue
    rows.append({
        "run": path.parent.name,
        "best_epoch": int(hist.loc[hist["dev_f1_weighted"].idxmax(), "epoch"]),
        "last_epoch": int(hist["epoch"].iloc[-1]),
        "best_dev_f1": round(hist["dev_f1_weighted"].max(), 4),
        "last_dev_f1": round(hist["dev_f1_weighted"].iloc[-1], 4),
    })
if rows:
    print("\nWhere dev F1 peaks (best_epoch < last_epoch means the reported score is past the peak):")
    display(pd.DataFrame(rows).sort_values("best_dev_f1", ascending=False))


## 7. Inspect the best run in detail

In [ ]:
best = df.sort_values(["hard_cons", "weighted_f1"], ascending=False).iloc[0]
run_dir = Path(best["predictions"]).parent
scores = json.loads((run_dir / "official_scores" / "scores.json").read_text(encoding="utf-8"))

print("Run         :", run_dir.name)
print("Backbone    :", best["model"])
print("Symmetric   :", bool(best.get("symmetric", False)))
print("Freeze      :", int(best["freeze_layers"]),
      f"bottom layers of {MODEL_LAYERS.get(best['model'], '?')}")
print("Trainable   :", best["trainable_pct"], "% of parameters")
print("Weighted F1 :", round(scores["weighted_f1"], 4))
print("SoftCons    :", round(scores["soft_cons"], 4))
print("HardCons    :", round(scores["hard_cons"], 4))

print("\nPer-label metrics:")
display(pd.DataFrame(scores["classification"]["per_label"]).T)

print("\nConfusion matrix (rows = gold, cols = predicted):")
display(pd.DataFrame(scores["classification"]["confusion_matrix"]))

print("\nSample pair errors (first 10):")
display(pd.DataFrame(scores["consistency"]["pair_errors"]).head(10))


## 8. (Optional) Compare against the full fine-tune baseline

`baseline_modernbert.ipynb` writes `results/baselines_modernbert/baseline_scores.csv`. If that file exists, this cell puts full fine-tuning next to the frozen runs.

In [ ]:
baseline_csv = OUTPUT_ROOT / "results" / "baselines_modernbert" / "baseline_scores.csv"
if baseline_csv.exists():
    base = pd.read_csv(baseline_csv)
    current_model = df.sort_values(["hard_cons", "weighted_f1"], ascending=False).iloc[0]["model"]
    base = base[(base["track"].astype(int) == TRACK) & (base["model"] == current_model)]
    if base.empty:
        print("No track", TRACK, current_model, "rows in", baseline_csv)
    else:
        reference = pd.Series({
            "model": f"{current_model} (full FT)",
            "freeze_layers": -1,
            "trainable_pct": 100.0,
            "weighted_f1_mean": base["weighted_f1"].mean(),
            "weighted_f1_std": base["weighted_f1"].std(),
            "soft_cons_mean": base["soft_cons"].mean(),
            "hard_cons_mean": base["hard_cons"].mean(),
            "n": len(base),
        })
        compare = pd.concat([summary, reference.to_frame().T], ignore_index=True)
        display(compare[["model", "freeze_layers", "trainable_pct", "n",
                         "weighted_f1_mean", "weighted_f1_std",
                         "soft_cons_mean", "hard_cons_mean"]].round(4))
else:
    print("No full-FT baseline found at", baseline_csv)
    print("Run notebooks/baseline_modernbert.ipynb (track 1) to generate it.")


## 9. (Optional) Tunable single run

Try one configuration without going through the whole grid. Change the three variables, then run the cell.

In [ ]:
MODEL_KEY = ACTIVE_MODELS[0]
FREEZE = FREEZE_VALUES[0]
SEED = SEEDS[0]

row = run_experiment(MODEL_KEY, MODELS[MODEL_KEY], FREEZE, SEED, overwrite=True)
pd.Series(row).to_frame().T


## 10. Build the CodaBench submission ZIP (dev format check)

CodaBench's rules for this task:

- one ZIP, prediction files **directly at the archive root** (no folder inside)
- filename must be exactly `track1_predictions.csv` (`track2_...`, `track3_...`, `track4_...`)
- header must be exactly `instance_id,label`
- one prediction per instance of the active phase, no duplicates, no unknown IDs
- labels drawn from the fixed 4-value set
- **nothing else in the archive** - no `.DS_Store`, no `__MACOSX/`, no README

A single-track ZIP is accepted: track 1 gets Weighted F1 / SoftCons / HardCons, tracks 2-4 show
`-1.0`, and `macro4_*` stays `-1.0` until all four files are in the same archive.

This takes a run's dev predictions, reorders the rows to the dev template, writes
`track1_predictions.csv`, and zips it with the file at the root. Use it in the
**Development phase** - dev instance IDs are what that phase expects.

> For the final submission, point `TRACK_RUNS` at a run evaluated on the unlabeled test CSV and
> swap the template for `*_test_track1_submission_template.csv`.


In [ ]:
import hashlib
import zipfile

from evaluation_functions import ALL_LABELS
from evaluation_functions.io import read_prediction_file

# ---- choose what goes into the ZIP ---------------------------------------
best_row = df.sort_values(["hard_cons", "weighted_f1"], ascending=False).iloc[0]
# "best" = the epoch with the highest dev F1 (from training_history.json).
# Use -1 for the last epoch, 0 for the first, 2 for the third ...
EXPORT_EPOCH = "best"

# track -> (model_key, freeze_bottom_layers, seed). Add more tracks here later.
TRACK_RUNS = {
    1: (best_row["model"], int(best_row["freeze_layers"]), int(best_row["seed"])),
}

STAGE = RESULTS_DIR / "submission_dev"
shutil.rmtree(STAGE, ignore_errors=True)
STAGE.mkdir(parents=True, exist_ok=True)
zip_path = RESULTS_DIR / "dico_nli_dev_submission.zip"
zip_path.unlink(missing_ok=True)

manifest = []
for track, (model_key, freeze, seed) in TRACK_RUNS.items():
    key = run_key(model_key, freeze, seed)
    epoch_preds = sorted((RESULTS_DIR / key).glob("predictions_dev_*.csv"))
    if not epoch_preds:
        raise FileNotFoundError(f"{key}: no predictions_dev_*.csv - run section 3 first.")
    if EXPORT_EPOCH == "best":
        chosen = best_row.get("best_epoch")
        if chosen is None or pd.isna(chosen):
            print(f"warning: {key} has no best_epoch recorded - exporting the last epoch")
            source = epoch_preds[-1]
        else:
            source = epoch_preds[int(chosen) - 1]
    else:
        source = epoch_preds[EXPORT_EPOCH]

    template = pd.read_csv(
        DATA_ROOT / "dev" / f"dico_nli_dev_track{track}_submission_template.csv", dtype=str
    )
    preds = pd.read_csv(source, dtype=str)

    if not preds["instance_id"].is_unique:
        raise ValueError(f"track {track}: duplicated instance_id in {source.name}")
    missing = set(template["instance_id"]) - set(preds["instance_id"])
    unknown = set(preds["instance_id"]) - set(template["instance_id"])
    if missing or unknown:
        raise ValueError(f"track {track}: {len(missing)} missing / {len(unknown)} unknown instance_ids")

    # Reorder to the template. CodaBench does not require it, but it makes the
    # upload reproducible and diffable against the template.
    preds = preds.set_index("instance_id").loc[template["instance_id"]].reset_index()

    csv_name = f"track{track}_predictions.csv"          # mandated filename
    dst = STAGE / csv_name
    preds.to_csv(dst, index=False, lineterminator="\n")  # header: instance_id,label

    manifest.append({
        "track": track,
        "file": csv_name,
        "run": key,
        "epoch_selection": str(EXPORT_EPOCH),
        "source": source.name,
        "rows": len(preds),
        "labels": preds["label"].nunique(),
        "sha256": hashlib.sha256(dst.read_bytes()).hexdigest()[:16],
    })

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for item in manifest:
        archive.write(STAGE / item["file"], arcname=item["file"])   # arcname has no directories

print(f"Wrote {zip_path} ({zip_path.stat().st_size:,} bytes)")
display(pd.DataFrame(manifest))


In [ ]:
# Re-open the ZIP and check it the way CodaBench does.
ACCEPTED = re.compile(r"^track[1-4]_predictions\.csv$")

with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
    print("ZIP entries:", names)

    assert names, "ZIP is empty"
    assert all("/" not in n and "\\" not in n for n in names), "files must sit at the ZIP root"
    assert all(ACCEPTED.match(n) for n in names), f"unexpected file(s): {names}"
    assert not any(n.startswith("__MACOSX") or n.endswith(".DS_Store") for n in names)

    for name in names:
        track = int(name[5])
        staged = STAGE / name
        assert archive.read(name) == staged.read_bytes(), f"{name}: ZIP content differs from staged file"

        records = read_prediction_file(staged)          # official header + label validation
        ids = [record.instance_id for record in records]
        assert len(ids) == len(set(ids)), f"{name}: duplicated instance_id"

        template = pd.read_csv(
            DATA_ROOT / "dev" / f"dico_nli_dev_track{track}_submission_template.csv", dtype=str
        )
        assert set(ids) == set(template["instance_id"]), f"{name}: instance_id mismatch"
        assert {record.label for record in records} <= set(ALL_LABELS), f"{name}: invalid label"

        first_lines = staged.read_text(encoding="utf-8").splitlines()[:2]
        print(f"\n{name}: {len(records)} predictions")
        print(f"  header : {first_lines[0]}")
        print(f"  first  : {first_lines[1]}")
        counts = pd.Series([record.label for record in records]).value_counts()
        print(f"  labels : {counts.to_dict()}")

print(f"\nOK - upload this archive to CodaBench: {zip_path}")
print("Single-track ZIP: track 1 is scored; tracks 2-4 and macro4_* show -1.0.")


## 11. What to look for in the results

- If **`trainable_pct` is ~27%** instead of ~0.8%/4%/7.5%, `--freeze_embeddings` did not reach
  `main.py`.
- If **`weighted_f1_best_mean` is well above `weighted_f1_last_mean`**, the run is overfitting
  before the epoch budget ends: lower `NUM_EPOCHS` or raise `DROPOUT`. Section 10 exports the best
  epoch by default, so the ZIP already uses the earlier checkpoint.
- If **`budget_limited_share` is 1.0** (or the progress line prints `best epoch is the last one`),
  the opposite happened: the curve was still rising when training ended. Raise `NUM_EPOCHS` - there
  is nothing to fix in the model.
- If **train loss barely moves** (it stalls near `ln(4) = 1.386`, or near the label-prior 1.36),
  the setting underfits: lower `FREEZE_VALUES` (12/14), raise `LR`, or keep
  `FREEZE_EMBEDDINGS = False` for the top layers to adapt the token representations.
- With a 660-row dev set, differences under ~1 F1 point are noise - compare the 5-seed means,
  not individual runs.
